In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

np.random.seed(0)

samples = 3000

a = np.random.uniform(-2,2,samples)
b = np.random.uniform(-2,2,samples)

data = np.column_stack((a,b))

labels = np.zeros(samples)

for i in range(samples):
    if a[i]**2 + b[i]**2 > 1.5:
        labels[i] = 1
    else:
        labels[i] = 0

labels = labels.reshape(-1,1)



In [ ]:
train_X,temp_X,train_y,temp_y = train_test_split(
    data,labels,test_size=0.3,random_state=42,shuffle=True
)

val_X,test_X,val_y,test_y = train_test_split(
    temp_X,temp_y,test_size=0.5,random_state=42,shuffle=True
)

print("Train size:",train_X.shape[0])
print("Val size:",val_X.shape[0])
print("Test size:",test_X.shape[0])



In [ ]:
def sigmoid_fn(z):
    return 1/(1+np.exp(-z))

def sigmoid_grad(a):
    return a*(1-a)

def relu_fn(z):
    return np.maximum(0,z)

def relu_grad(z):
    g = np.zeros_like(z)
    g[z>0] = 1
    return g



In [ ]:
def loss_fn(y_true,y_pred):

    eps = 1e-8
    y_pred = np.clip(y_pred,eps,1-eps)

    value = -np.mean(y_true*np.log(y_pred) + (1-y_true)*np.log(1-y_pred))

    return value


def accuracy_fn(y_true,y_pred):

    y_binary = (y_pred>=0.5).astype(int)
    return np.mean(y_binary == y_true)



In [ ]:
def init_layers(structure):

    W_list=[]
    b_list=[]

    for i in range(len(structure)-1):

        W = np.random.randn(structure[i],structure[i+1]) * 0.1
        b = np.zeros((1,structure[i+1]))

        W_list.append(W)
        b_list.append(b)

    return W_list,b_list



In [ ]:
def forward_pass(X,W_list,b_list,act_type):

    acts=[X]
    z_list=[]

    for i in range(len(W_list)-1):

        z = acts[-1] @ W_list[i] + b_list[i]
        z_list.append(z)

        if act_type=="sigmoid":
            a = sigmoid_fn(z)
        else:
            a = relu_fn(z)

        acts.append(a)

    z = acts[-1] @ W_list[-1] + b_list[-1]
    z_list.append(z)

    a = sigmoid_fn(z)
    acts.append(a)

    return acts,z_list



In [ ]:
def backward_pass(y,acts,z_list,W_list,act_type):

    gradW=[]
    gradb=[]

    m = y.shape[0]

    dz = acts[-1] - y

    for i in reversed(range(len(W_list))):

        prev = acts[i]

        dW = (prev.T @ dz)/m
        db = np.sum(dz,axis=0,keepdims=True)/m

        gradW.insert(0,dW)
        gradb.insert(0,db)

        if i != 0:

            da = dz @ W_list[i].T

            if act_type=="sigmoid":
                dz = da * sigmoid_grad(acts[i])
            else:
                dz = da * relu_grad(z_list[i-1])

    return gradW,gradb



In [ ]:
def train_network(structure,activation,optimizer,lr=0.1,epochs=100):

    W_list,b_list = init_layers(structure)

    velW=[np.zeros_like(W) for W in W_list]
    velb=[np.zeros_like(b) for b in b_list]

    beta = 0.9

    train_loss=[]
    val_loss=[]
    train_acc=[]
    val_acc=[]

    first_grad=[]
    last_grad=[]

    for e in range(epochs):

        acts,zs = forward_pass(train_X,W_list,b_list,activation)

        loss = loss_fn(train_y,acts[-1])
        acc = accuracy_fn(train_y,acts[-1])

        gW,gb = backward_pass(train_y,acts,zs,W_list,activation)

        first_grad.append(np.linalg.norm(gW[0]))
        last_grad.append(np.linalg.norm(gW[-1]))

        for i in range(len(W_list)):

            if optimizer=="sgd":

                W_list[i] = W_list[i] - lr*gW[i]
                b_list[i] = b_list[i] - lr*gb[i]

            else:

                velW[i] = beta*velW[i] + lr*gW[i]
                velb[i] = beta*velb[i] + lr*gb[i]

                W_list[i] = W_list[i] - velW[i]
                b_list[i] = b_list[i] - velb[i]

        val_act,_ = forward_pass(val_X,W_list,b_list,activation)

        v_loss = loss_fn(val_y,val_act[-1])
        v_acc = accuracy_fn(val_y,val_act[-1])

        train_loss.append(loss)
        val_loss.append(v_loss)
        train_acc.append(acc)
        val_acc.append(v_acc)

        if e%10==0:
            print("Epoch:",e,"Train Loss:",loss,"Val Loss:",v_loss)

    return (W_list,b_list,train_loss,val_loss,train_acc,val_acc,first_grad,last_grad)



In [ ]:
layers=[2,8,8,8,1]

result=train_network(layers,activation="sigmoid",optimizer="sgd",lr=0.1,epochs=100)

(W_list,b_list,train_loss,val_loss,train_acc,val_acc,first_grad,last_grad)=result

plt.figure()
plt.plot(train_loss)
plt.plot(val_loss)
plt.title("Loss vs Epoch")
plt.legend(["Train","Validation"])
plt.show()

plt.figure()
plt.plot(train_acc)
plt.plot(val_acc)
plt.title("Accuracy vs Epoch")
plt.legend(["Train","Validation"])
plt.show()

test_act,_ = forward_pass(test_X,W_list,b_list,"sigmoid")

print("Final Test Loss:",loss_fn(test_y,test_act[-1]))
print("Final Test Accuracy:",accuracy_fn(test_y,test_act[-1]))

plt.figure()
plt.plot(first_grad)
plt.plot(last_grad)
plt.legend(["First Layer","Last Layer"])
plt.title("Gradient Norm Comparison")
plt.show()

